In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow.models import infer_signature

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [0]:
catalog = "workspace"
schema = "financial_sentiment"
table_prefix = f"{catalog}.{schema}"

experiment_name = "/Users/jezapataf@eafit.edu.co/financial_sentiment_mlops"
mlflow.set_experiment(experiment_name)

In [0]:
df = spark.table(f"{table_prefix}.gold_financial_sentiment_training").toPandas()

print("Total registros:", len(df))
print(df["split"].value_counts())
print(df["label_normalized"].value_counts())

In [0]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()

if len(train_df) == 0:
    raise Exception("No hay datos de entrenamiento.")

if len(val_df) == 0:
    raise Exception("No hay datos de validación.")

X_train = train_df[["text_clean"]]
y_train = train_df["label_normalized"]

X_val = val_df[["text_clean"]]
y_val = val_df["label_normalized"]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

In [0]:
def select_text_column(df):
    return df["text_clean"]

model = Pipeline([
    ("selector", FunctionTransformer(select_text_column, validate=False)),
    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

In [0]:
with mlflow.start_run(run_name="baseline_tfidf_logreg_with_signature") as run:
    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    acc = accuracy_score(y_val, preds)
    macro_f1 = f1_score(y_val, preds, average="macro")
    weighted_f1 = f1_score(y_val, preds, average="weighted")

    mlflow.log_param("model_type", "tfidf_logistic_regression")
    mlflow.log_param("max_features", 10000)
    mlflow.log_param("ngram_range", "1,2")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("input_column", "text_clean")

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("macro_f1", macro_f1)
    mlflow.log_metric("weighted_f1", weighted_f1)

    report = classification_report(y_val, preds, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_path = "/tmp/classification_report.csv"
    report_df.to_csv(report_path)
    mlflow.log_artifact(report_path)

    input_example = X_val.head(5)
    output_example = model.predict(input_example)

    signature = infer_signature(
        model_input=input_example,
        model_output=output_example
    )

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example
    )

    print("Run ID:", run.info.run_id)
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Weighted F1:", weighted_f1)

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
run = client.get_run(run.info.run_id)

print("Último run:", run.info.run_id)